# Read me

#### License
This code is licensed under the MIT License. You are free to use, modify, and distribute it, provided you cite the original paper [1].

#### Dependencies
The tests were performed with the following versions:
- Python 3.12.7
- Pandas 2.2.2
- NumPy 1.26.4
- Aiphad 1.0.1
- TC-Python 2024.2.32679.198 (Licence for Thermo-Calc [2] required)
 
#### Description


This code implements a multi-step computational methodology developed in [1]:

- Primary crystallization phases are calculated via Thermo-Calc [2] for compositions strategically selected through active learning. <br>
- Phase labels are propagated and primary crystallization regions are determined using AIPHAD [3]. <br>
- Eutectic candidates are identified as intersections where *n* phases meet, with *n* corresponding to the number of elements in the system. <br>
- The identified compositions serve as starting points for Solidus descent [1] to confirm, refine, or refute the eutectic point. <br>



This code emphasizes scientific reproducibility of the method developed in [1] rather than adherence to programming best practices.

##### References
[1] K. Firdová, K. E. Borowski, R. Tamura, J.-M. Joubert, G. Deffrennes; Methods for efficient eutectic screening in CALPHAD databases <br>
[2] The Thermo-Calc Software TCAL09 Al-based Alloy Database https://thermocalc.com/products/databases/aluminum-based-alloys/ <br>
[3] R. Tamura, H. Morito, G. Deffrennes, M. Naito, Y. Nose, T. Abe, K. Terayama; AIPHAD, an active learning web application for visual understanding of phase diagrams, Commun. Mater. 5 (2024) https://doi.org/https://doi.org/10.1038/s43246-024-00580-7

# Requirements


In [3]:
import pandas as pd
import numpy as np
from aiphad import pdc_sampler
import os
import time
from tc_python import *

## Input parameters

In [5]:
#System
#elements = [["Si", "Mg", "Zn", "Cu"]]
elements = [["Cu", "Mg", "Zn"]]

In [6]:
database = 'TCAL9'
delta_T = 0.01 #The decrease in temperature in K from the liquidus temperature to find the primary crystallization phase
T0 = 600 # Starting temperature for the compute-transition strategy
T0_l = [300,600,900,1500,2500]

max_iterations=120
max_compute_transition_trials=3
significant_digits=5
melting_point_dic = {'Ag': 1234.93, 'Al': 933.47, 'As': 1090.0, 'Au': 1337.33, 'B': 2348.0, 'Ba': 1000.0, 'Be': 1560.0, 'Bi': 544.55, 'C': 4765.29, 'Ca': 1115.0, 'Cd': 594.22, 'Ce': 1072.0, 'Co': 1767.99, 'Cr': 2179.99, 'Cs': 301.59, 'Cu': 1357.77, 'Dy': 1685.11, 'Er': 1802.0, 'Eu': 1095.0, 'Fe': 1810.96, 'Ga': 302.91, 'Gd': 1586.12, 'Ge': 1211.4, 'Hf': 2506.0, 'Hg': 234.32, 'Ho': 1747.0, 'In': 429.75, 'Ir': 2719.0, 'K': 336.53, 'La': 1193.0, 'Li': 453.6, 'Lu': 1936.0, 'Mg': 923.0, 'Mn': 1519.0, 'Mo': 2896.01, 'Na': 370.87, 'Nb': 2750.0, 'Nd': 1289.0, 'Ni': 1728.25, 'Os': 3306.0, 'P': 852.37, 'Pb': 600.61, 'Pd': 1828.0, 'Pr': 1204.0, 'Pt': 2041.5, 'Pu': 913.0, 'Rb': 312.46, 'Re': 3459.0, 'Rh': 2237.0, 'Ru': 2607.0, 'S': 388.36, 'Sb': 903.78, 'Sc': 1814.0, 'Se': 494.0, 'Si': 1687.0, 'Sm': 1345.0, 'Sn': 505.08, 'Sr': 1050.0, 'Ta': 3290.0, 'Tb': 1632.0, 'Tc': 2430.0, 'Te': 722.66, 'Ti': 1940.99, 'Tl': 577.0, 'Tm': 1818.0, 'U': 1407.99, 'V': 2183.0, 'W': 3694.9, 'Y': 1795.15, 'Yb': 1097.0, 'Zn': 692.68, 'Zr': 2127.86}


In [7]:
n_active_points= 200

composition_grid = np.linspace(0, 1, 5)
step=26 #every 4%
neighbour_step_a = 1/(step-1)
n_phases=len(elements[0])-1
page_type=None

# TC-Python functions

#### To obtain liquidus, solidus and primary crystallization phases


In [9]:
def compute_liquidus(element_list, composition_list, T0):
    
    #'Compute-Transition' calculation
    calc = system.with_single_equilibrium_calculation()
    try: #Try and except prevent the calculation from crashing if one calculation doesn't converge
        calc.set_phase_to_entered("*") \
            .set_condition(ThermodynamicQuantity.temperature(), T0) \
            .disable_global_minimization() #disable_global_minimization: this help this intial calculation to converge (more info p.1597 "GLOBAL_MINIMIZATION" of the Thermo-calc Documentation set)             
        for j in range(len(element_list)): #This should not be put before entering the first conditions, or we will get an error !
            if composition_list[j] == 0:
                calc.set_component_to_suspended(element_list[j], True)
            else:
                calc.set_condition(ThermodynamicQuantity.mole_fraction_of_a_component(element_list[j]), composition_list[j])       
        calc_result = calc.calculate()

        T_incr = 0
        while T0+T_incr <= 5000: #To obtain the liquidus with 'compute-transition', we must start where we are 100% liquid ! We increase the temperature until this is true or the temperature is too high...  
            if calc_result.get_value_of('NPM(LIQUID)') == 1: 
                calc_result = calc.run_poly_command('c-t any T -1').calculate() #We remove T as a constrain and we are looking for any phase to form instead of the liquid
                return calc_result.get_value_of(ThermodynamicQuantity.temperature())
                break
            calc_result = calc.set_condition(ThermodynamicQuantity.temperature(), T0+T_incr).calculate()
            T_incr += 200
    except Exception as e:
        print('Could not calculate. Trying with LIQ = FIX1')
        
        #'LIQ = FIX 1' calculation if the compute transition calculation fails...
        T_list = []
        for T0 in T0_l:
            try:   
                calc.set_phase_to_entered("*") \
                    .set_condition(ThermodynamicQuantity.temperature(), T0) \
                    .disable_global_minimization() #disable_global_minimization: this help this intial calculation to converge (more info p.1597 "GLOBAL_MINIMIZATION" of the Thermo-calc Documentation set)             
                for j in range(len(element_list)): #This should not be put before entering the first conditions, or we will get an error !
                    if composition_list[j] == 0:
                        calc.set_component_to_suspended(element_list[j], True)
                    else:
                        calc.set_condition(ThermodynamicQuantity.mole_fraction_of_a_component(element_list[j]), composition_list[j])       
                calc.calculate() #This is an initial calculation to helps the LIQ = FIX1 calculation converge

                calc_result = (calc.remove_condition(ThermodynamicQuantity.temperature()).set_phase_to_fixed('LIQUID', 1)  \
                               .enable_global_minimization().calculate())
                T_list.append(calc_result.get_value_of(ThermodynamicQuantity.temperature()))
                
            except Exception as e:
                print('Could not calculate. Continuing with next...')            
                
        if T_list:
            return(min(T_list))  
        
        else:
            return 3000



def get_primary_crystallisation_phase(element_list, composition_list, T0_):

    calc = system.with_single_equilibrium_calculation()
    try: #Try and except prevent the calculation from crashing if one calculation doesn't converge
        calc.set_phase_to_entered("*") \
                .set_condition(ThermodynamicQuantity.temperature(), T0_) \
                .disable_global_minimization() #disable_global_minimization: this help this intial calculation to converge (more info p.1597 "GLOBAL_MINIMIZATION" of the Thermo-calc Documentation set)             
        for j in range(len(element_list)): #This should not be put before entering the first conditions, or we will get an error !
            if composition_list[j] == 0:
                calc.set_component_to_suspended(element_list[j], True)
            else:
                calc.set_condition(ThermodynamicQuantity.mole_fraction_of_a_component(element_list[j]), composition_list[j])       
        calc_result = calc.calculate()
        
        if len(calc_result.get_stable_phases())==1 and "LIQ" in str(calc_result.get_stable_phases()):
            T_deltas = [0.01,0.02,1,5,10,20,50,100]
            print(composition_list[j])
            print(T0_)
            for delta in T_deltas:            
                print('No solid phase found. Trying with delta = {})'.format(delta_T + delta))
                calc.set_condition(ThermodynamicQuantity.temperature(), T0_-delta).enable_global_minimization() 
                calc_result = calc.calculate()
                if len(calc_result.get_stable_phases())!=1 or "LIQ" not in str(calc_result.get_stable_phases()):
                    break

        return calc_result.get_stable_phases()        
    except Exception as e:
        print('Could not calculate neighbouring composition.')

######################################################################################################################################

def get_calc_results(calc_result, element_list):
    solidus_temperature = round(calc_result.get_value_of(ThermodynamicQuantity.temperature()),significant_digits-3)
    liquid_composition_l = []
    for element in element_list:
        liquid_composition_l.append(round(calc_result.get_value_of('X(LIQUID,'+element+')'),significant_digits))
    number_of_phases = len(calc_result.get_stable_phases())+1
    return solidus_temperature, liquid_composition_l, number_of_phases


def compute_solidus_and_liq_compo(element_list, composition_list):
    
    #'Compute-Transition' calculation
    calc = system.with_single_equilibrium_calculation()
    try: #Try and except prevent the calculation from crashing if one calculation doesn't converge     
        
    # STEP 1: Make an initial calculation at a temperature where there is no liquid

        #Find the initial temperature which is set as the average melting point between elements
        melting_point_l = []
        for element in element_list:
            melting_point=melting_point_dic[element]
            melting_point_l.append(melting_point)
        T0 = round(sum(a * b for a, b in zip(melting_point_l, composition_list)),0)

        calc.set_phase_to_entered("*") \
            .set_condition(ThermodynamicQuantity.temperature(), T0) \
            .disable_global_minimization() #disable_global_minimization: this help this intial calculation to converge (more info p.1597 "GLOBAL_MINIMIZATION" of the Thermo-calc Documentation set)             
        
        #The first element of the list is the balance, we have to suspend the elements until the balance is not zero
        #That is because there are convergence problems if we enter an element with a composition of zero
        el_list = element_list
        compo_list = composition_list
        while compo_list[0] == 0:
            calc.set_component_to_suspended(el_list[0], True)
            el_list = el_list[1:]
            compo_list = compo_list[1:]            
        #We remove the first element of the list because it's the balance in the calculation
        el_list = el_list[1:]
        compo_list = compo_list[1:]
        for j in range(len(el_list)): #This should not be put before entering the first conditions, or we will get an error !
            if compo_list[j] == 0:
                calc.set_component_to_suspended(el_list[j], True)
            else:
                calc.set_condition(ThermodynamicQuantity.mole_fraction_of_a_component(el_list[j]), compo_list[j])       
        calc_result = calc.calculate()
        
        while round(calc_result.get_value_of('NPM(LIQUID)'),4)>0.000: #Check that we didn't go above the solidus, i.e., the fraction of liquid is zero
            T0=T0-100
            if T0>298.15:
                calc.set_condition(ThermodynamicQuantity.temperature(), T0)
                calc_result = calc.calculate()
            else:
                T0 = 298.15
                calc.set_condition(ThermodynamicQuantity.temperature(), T0)
                calc_result = calc.calculate()
                break
            
    # STEP 2: Perform a compute transition calculation
        if round(calc_result.get_value_of('NPM(LIQUID)'),4) == 0.000: #Check that we didn't go above the solidus, i.e., the fraction of liquid is zero
            calc_result = calc.run_poly_command('c-t LIQUID T').enable_global_minimization().calculate() #We remove T as a constrain and we are looking for any phase to form instead of the liquid
            return get_calc_results(calc_result, element_list)
        else:
            print('Calculation failed: the solidus seems to be lower than 300K!')
            return None, None, None
    
    except Exception as e: #If the solidus calculation didn't work, we try to find a transition closer to the liquidus
        n_trials = 0
        while n_trials < max_compute_transition_trials:
            print('Could not calculate. Trying from a transition closer to the solidus')
            n_trials = n_trials+1
            
            try:
                calc_result = calc.run_poly_command('c-t any T 1').enable_global_minimization().calculate() #We remove T as a constrain and we are looking for any phase to form instead of the liquid
            except Exception as e:
                print('Calculation failed: the calculation to find a transition closer to the liquidus diverged.')
                try:
                    calc_result = (calc.remove_condition(ThermodynamicQuantity.temperature()).set_phase_to_fixed('LIQUID', 0)  \
                                   .enable_global_minimization().calculate())
                    return get_calc_results(calc_result, element_list)  
                except Exception as e:
                    print('Last try with the LIQ = FIX 0 method also failed. Going next.')
                    return None, None, None
        
            if round(calc_result.get_value_of('NPM(LIQUID)'),4) == 0.000: #Check that we didn't go above the solidus, i.e., the fraction of liquid is zero
                try:
                    calc_result = calc.run_poly_command('c-t LIQUID T').enable_global_minimization().calculate() #We remove T as a constrain and we are looking for any phase to form instead of the liquid
                    return get_calc_results(calc_result, element_list)       
                except Exception as e:
                    pass
            else:
                print('Calculation failed: we went above the solidus')
                try:
                    calc_result = (calc.remove_condition(ThermodynamicQuantity.temperature()).set_phase_to_fixed('LIQUID', 0)  \
                                   .enable_global_minimization().calculate())
                    return get_calc_results(calc_result, element_list)  
                except Exception as e:
                    print('Last try with the LIQ = FIX 0 method also failed. Going next.')
                    return None, None, None        
        
        print('Calculation failed: we could not find the solidus after '+str(max_compute_transition_trials)+' compute-transition trials')
        try:
            calc_result = (calc.remove_condition(ThermodynamicQuantity.temperature()).set_phase_to_fixed('LIQUID', 0)  \
                           .enable_global_minimization().calculate())
            return get_calc_results(calc_result, element_list)  
        except Exception as e:
            print('Last try with the LIQ = FIX 0 method also failed. Going next.')
            return None, None, None


def find_eutectics(element_list, composition_list):
    solidus_temperature, previous_liquid_composition, nb_of_phases = compute_solidus_and_liq_compo(element_list, composition_list)
    if solidus_temperature == None:
        return None, None, None, 0, 'solidus_temperature = None'
    else:
        n=0
        while n < max_iterations:
            if all(c >= 0.01 for c in previous_liquid_composition):
                solidus_temperature, new_liquid_composition, nb_of_phases = compute_solidus_and_liq_compo(element_list, previous_liquid_composition)
                if solidus_temperature == None:
                    return None, None, None  , n, 'solidus_temperature = None'         
                else:
                    if previous_liquid_composition == new_liquid_composition:
                        if (nb_of_phases != None) and (nb_of_phases < 3) :  # If there is only one phase beside the liquid
                            print('This is a congruent melting compound, a pure element, or a liquidus minimum above a single solid solution')
                            # N.B. To avoid convergence to special invariant points, such as a minima on a monovariant line,
                            # use nb_of_phases < len(element_list) + 1 instead of nb_of_phases < 3.
                            return None, None, None, n, 'This is a congruent melting compound or a pure element'
                        else:            
                            return solidus_temperature, new_liquid_composition, nb_of_phases, n, 'ok'
                    else:
                        previous_liquid_composition = new_liquid_composition
                        n=n+1
            else:
                print('Composition smaller than 1%')
                return None, None, None, n, 'Composition smaller than 1%'
                
        print('Maximum number of iterations reached')
        return None, None, None, n, 'Maximum number of iterations reached'


# Primary crystallization phases predictions with active learning

In [11]:
from Aiphad_dataframes_functions import compo_grid,  save_df_liquidus_phases, prepare_df_for_aiphad, update_df
from Aiphad_dataframes_functions import update_aiphad_df

In [12]:
def aiphad_loop(k,df,elements,step, page_type, estimation='LS'):
    print(k)
    proposals_l=[]
    proposal_to_skip=[]
    for i in range(k):
        print('********')
        print(i)
        #df_aiphad = pd.read_csv('-'.join(elements)+'data_aiphad_input.csv')
        pdc  = pdc_sampler(page_type = page_type, input_data = os.getcwd()+'\\'+'-'.join(elements)+'data_aiphad_input.csv',estimation = estimation, sampling = "EA", proposal = 10)
        pdc.fit()
        pdc.us()
        proposals_X = pdc.proposals_X
        if len(proposal_to_skip)>0:
            i=0
            while proposals_X[i] in proposal_to_skip:
                i=i+1
            proposals_X = proposals_X[i] 
            print(i)

        else:
            proposals_X = proposals_X[0]
           
        liquidus_temperature = compute_liquidus(elements[1:], proposals_X[1:], T0) 
        calc_T = liquidus_temperature-delta_T
        primary_phase = get_primary_crystallisation_phase(elements[1:], proposals_X[1:], calc_T) 
        print(primary_phase)
        if primary_phase==None:
            continue
        if len(primary_phase) == 1 and 'LIQ' in primary_phase[0]:
            proposal_to_skip.append(proposals_X)
            #df_aiphad = prepare_df_for_aiphad(elements, step, proposal_to_skip)
            continue
        df,df_aiphad = update_aiphad_df(df, liquidus_temperature, primary_phase, proposals_X, elements )
        proposals_l.append(proposals_X)

    return pdc, df, proposals_l

# Eutectics candidates search functions

In [14]:
from Aiphad_candidates_functions import build_df_estimate, find_candidates, find_final_candidates, process_candidates

# Analysis

In [16]:
%%time

elements = elements[0]
df = compo_grid(composition_grid, elements)


start_time = time.time()  # Record the start time

tc_python.server.start_api_server()
system = SetUp().select_database_and_elements(database, elements).deselect_phase('GAS').get_system()


liquidus_temperature_l = [] #List to store the resuts of batch calculations to later create a dataframe
primary_cryst_phase_l = [] #List to store the resuts of batch calculations to later create a dataframe

for row in range(len(df)): #For every composition in our dataframe
    liquidus_temperature = compute_liquidus(elements[1:], df.iloc[row].tolist()[1:], T0) # [1:] because we don't take the first element which is the balance to have a global composition of 100%!   
    liquidus_temperature_l.append(liquidus_temperature)
    calc_T = liquidus_temperature-delta_T
    primary_phase = get_primary_crystallisation_phase(elements[1:], df.iloc[row].tolist()[1:], calc_T) # [1:] because we don't take the first element which is the balance to have a global composition of 100%!    
    primary_cryst_phase_l.append(primary_phase)

indices_none = [i for i, phs in enumerate(primary_cryst_phase_l) if phs is None]
if len(indices_none) !=0:
    liquidus_temperature_l[indices_none[0]] = 'none'
    primary_cryst_phase_l[indices_none[0]] = 'none' 

save_df_liquidus_phases(df=df, elements=elements, liquidus_temperature_l=liquidus_temperature_l,primary_cryst_phase_l=primary_cryst_phase_l)
df_aiphad = prepare_df_for_aiphad(df, elements, step)

pdc_, df, proposals = aiphad_loop(n_active_points,df,elements,step, page_type)


df_estimate = build_df_estimate(pdc_)
candidates = find_candidates(elements, df_estimate, neighbour_step_a+0.01)
df_candidates = process_candidates(candidates)
final_candidates = find_final_candidates(df_candidates, n_phases=n_phases)

print('Candidates on the intersection of n phases:')
print(len(final_candidates))
  


eutectic_temperature_l = []
eutectic_composition_l = []
nb_of_phases_l = []


df_candidates = df_estimate.iloc[final_candidates,:len(elements)].copy() 
df_candidates = df_candidates.replace(0.00, 0.05)
df_candidates = df_candidates.replace(1.00, 0.95)


for row in range(len(df_candidates)): #For every composition in our dataframe
    print(df_candidates.iloc[row].tolist())
    eutectic_temperature, eutectic_compo, nb_of_phases, it, er = find_eutectics(elements, df_candidates.iloc[row].tolist())
    print(eutectic_compo)
    eutectic_temperature_l.append(eutectic_temperature)
    eutectic_composition_l.append(eutectic_compo)
    nb_of_phases_l.append(nb_of_phases)


end_time = time.time()
total_time = end_time - start_time

tc_python.server.stop_api_server()
print(total_time)

        


15:37:05,614 [Thread-1] INFO  CalculationEngine: Starting Thermo-Calc 2024b, Java-version: 21.0.2
15:37:06,175 [Thread-1] INFO  CalculationEngine: Opening database::TCAL9::
15:37:07,131 [Thread-1] INFO  JavaWrapper: *** Invoking Gibbs Energy System v6 ***
15:37:09,126 [Thread-1] INFO  CalculationEngine:    'Yang YANG, Thermo-Calc Software AB, Sweden; Assessment of ELRS/THCD for
15:37:09,126 [Thread-1] INFO  CalculationEngine:        Ti alloys'
15:37:09,126 [Thread-1] INFO  CalculationEngine:    'A. Dinsdale, Calphad,15 317-425 (1991)'
15:37:09,126 [Thread-1] INFO  CalculationEngine:    'Y.Tang, in TCCU2, Thermo-Calc Software AB, Stockholm, Sweden'
15:37:09,126 [Thread-1] INFO  CalculationEngine:    'L. Kjellqvist, Thermo-Calc Software AB, Sweden 2012; Molar volumes'
15:37:09,126 [Thread-1] INFO  CalculationEngine:    'X.-G. Lu, M. Selleby and B. Sundman, CALPHAD, Vol. 29, 2005, pp. 68-89;
15:37:09,126 [Thread-1] INFO  CalculationEngine:        Molar volumes'
15:37:09,126 [Thread-1] INF

In [17]:
eutectic_composition_l

[None,
 None,
 None,
 None,
 None,
 None,
 [0.14324, 0.7714, 0.08536],
 [0.14324, 0.7714, 0.08536],
 None,
 [0.53056, 0.13364, 0.3358],
 [0.53056, 0.13364, 0.3358],
 None,
 None,
 None,
 [0.14324, 0.7714, 0.08536],
 None,
 None,
 None,
 None,
 [0.53056, 0.13364, 0.3358],
 None,
 None,
 None]